# 04 — Promote production version (`stage=prod`)

OpenShift AI Model Registry is the **production boundary**.

This notebook sets `stage=prod` on the selected version and clears it from any previous prod version.
That is the contract the Data Science Pipeline resolves (replacing MLflow `@prod` aliases).

In [ ]:
import os
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").exists() and (REPO_ROOT.parent / "src").exists():
    REPO_ROOT = REPO_ROOT.parent
    os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))

from src.registry_client import ModelRegistryClient

model_name = os.environ.get("REGISTERED_MODEL_NAME", "xray-detector")
version_id = os.environ.get("SELECTED_VERSION_ID")
if not version_id:
    raise SystemExit("Set SELECTED_VERSION_ID (from notebooks/03_export_register.ipynb)")

registry = ModelRegistryClient()
result = registry.promote_to_prod(model_name, version_id)
print("Promoted version:", result["promoted_version"].get("id"), result["promoted_version"].get("name"))
print("Cleared previous prod version ids:", result["cleared_version_ids"])

In [ ]:
resolved = registry.resolve_prod(model_name)
print("Pipeline contract (resolve prod):")
for key in [
    "registered_model_name",
    "version_id",
    "version_name",
    "uri",
    "storage_key",
    "storage_path",
    "mlflow_run_id",
]:
    print(f"  {key}: {resolved.get(key)}")

print()
print("Next: run pipelines/promote_deploy_canary.py (Data Science Pipelines)")
print("  action=leave-canary | promote | rollback")
print("Then validate with notebooks/05_infer_served.ipynb")